# Pulse

> Utilities for working with ECG/EKG data. 

In [ ]:
#| default_exp pulse

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import psycopg
import yaml
import urllib

import numpy as np
from pathlib import Path
from wavefm.signal import butterworth

import matplotlib.pyplot as plt
import seaborn as sns

from torch import nn, Tensor
import torch


In [ ]:
#| export
def connect_to_postgres(postgres_connection_path='/hpc/users/foxb02/.ecg_postgres.yaml' # Path of yaml file with configurations for postgres database
                        ):
    """
    Function to connect to a postgres database
    """
    with open(postgres_connection_path) as f:
        db_connect_params = yaml.safe_load(f)

    postgres_str = f"postgresql://{db_connect_params['username']}:{urllib.parse.quote_plus(db_connect_params['password'])}@{db_connect_params['host']}:{db_connect_params['port']}/{db_connect_params['database']}"
    conn = psycopg.connect(postgres_str)
    cursor = conn.cursor()
    return conn, cursor

def create_table_from_csv(cursor,
                          connection,
                          table_name:str, # Name of the table
                          table_query:str, # Table creation query 
                          csv_path:str # CSV path for table
                          ):
    """
    Function to create a temporary table in a database.

    Example
    --------
    CREATE TEMPORARY TABLE LVEF_DEID(
        deid_mrn CHARACTER VARYING(64),
        LVEF CHARACTER VARYING(64),
        LVEF_datetime TIMESTAMP WITHOUT TIME ZONE,
        LVEF_Class INTEGER
        )
    """
    cursor.execute(table_query)

    with open(csv_path, "r") as f:
        with cursor.copy(f"COPY {table_name} FROM STDIN DELIMITER ',' CSV HEADER" ) as copy:
            while data := f.read(100):
                copy.write(data)
    connection.commit()

In [ ]:
#| export
def decode_waveform(waveform_string:str # Waveform string
                    )->list: # List of int16 waveform values
    """
    Function to convert a waveform string from a buffer
    """
    return list(np.frombuffer(waveform_string, dtype='int16'))

def subset_waveform(waveform_array:list, # Array/list of waveform integer values
                    cutoff:int=5000, # Length of array subset
                    random_index:bool=False, # Indicator of whether to randomly subset the waveform at a random start index
                    random_cutoff:bool=False # Indicator of whether to randomly generator a cutoff value
                    )->list: # subsetted waveform array
    """
    Function to take a subset of a waveform array
    """
    if random_cutoff:
        cutoff = np.random.randint(0, len(waveform_array))
    if random_index and len(waveform_array) > cutoff:
        low_idx = np.random.randint(0, len(waveform_array) - cutoff)
        high_idx = low_idx + cutoff
    else:
        low_idx = 0
        high_idx = cutoff
    return waveform_array[low_idx:int(high_idx)]

def process_ecg(waveform_array:list, # Array/list of waveform integer values, 
                cutoff, # Length of array subset, 
                random_index:bool=True, # Indicator of whether to randomly subset the waveform at a random start index
                random_cutoff:bool=False # Indicator of whether to randomly generator a cutoff value
                )->list: # Processed waveform array
    """
    High level function to process ecg data
    """
    waveform_array = decode_waveform(waveform_array)
    waveform_array = subset_waveform(waveform_array, cutoff=cutoff, random_index=random_index, random_cutoff=random_cutoff)
    waveform_array = butterworth(waveform_array).astype(np.float32)
    return waveform_array

def generate_plot(waveform_array, 
                  figsize, 
                  dpi, 
                  save=False, 
                  save_file_path=None, 
                  return_figure=False
                  ):
    """
    Function to plot a 
    """
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, tight_layout=True)
    ax.set_axis_off()
    sns.lineplot(waveform_array, color='black', linewidth=0.5, ax=ax)
    plt.close()
    if save and save_file_path:
        save_file_path = Path(save_file_path)
        if not save_file_path.parent.exists():
            save_file_path.parent.mkdir(parents=True)
        fig.savefig(save_file_path, dpi=dpi, format='png', transparent=False)
    if return_figure:
        return fig, ax

In [ ]:
#| export
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: Tensor, shape [seq_len, batch_size, embedding_dim]
        """
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

class ECG_Model(nn.Module):
    def __init__(self, c_in, d_model, c_out, nhead, dim_ff, num_layers, compression_rate: int = 100, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=compression_rate, stride=compression_rate)
        #self.encoder = nn.Embedding(c_in//compression_rate, d_model)
        #self.pos_encoder = PositionalEncoding(d_model, dropout, max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )
        self.classifier = nn.Linear(d_model, c_out)
        self.d_model = d_model

    def forward(self, x):
        x = self.conv(x)
        #x = self.encoder(x) * math.sqrt(self.d_model)
        #x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        x = self.classifier(x)
        return x
    
class ECG_Model_TSAI(nn.Module):
    def __init__(self, c_in, d_model, c_out, n_head, dim_ff, n_layers, compression_channels: int = 32, compression_rate: int = 100, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=c_in, out_channels=compression_channels, kernel_size=compression_rate, stride=compression_rate)
        #self.conv = random_convolution(
        #self.encoder = nn.Embedding(c_in//compression_rate, d_model)
        #self.pos_encoder = PositionalEncoding(d_model, dropout, max_len)
        self.transformer = TransformerModel(c_in=compression_channels, c_out=c_out, d_model=d_model, n_head=n_head, d_ffn=dim_ff, dropout=dropout, n_layers=n_layers)
        self.d_model = d_model

    def forward(self, x):
        x = self.conv(x)
        #x = random_convolution(x, c_in=1, c_out=1, max_kernel_size=1000, mix = False, p = 0.8)
        #x = self.encoder(x) * math.sqrt(self.d_model)
        #x = self.pos_encoder(x)
        x = self.transformer(x)
#        x = x.mean(dim=1)
#        x = self.classifier(x)
        return x

class ECG_Model_TST(nn.Module):
    def __init__(self, c_in, c_out, seq_len, n_layers: int = 3, d_model: int = 128, n_heads: int = 16, d_ff: int = 256, dropout: float = 0.1, act = 'gelu', compression_channels: int = 32, compression_rate: int = 100):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=c_in, out_channels=compression_channels, kernel_size=compression_rate, stride=compression_rate)
        self.dropout1 = nn.Dropout(p=dropout, inplace=False)
        #self.conv = random_convolution(
        #self.encoder = nn.Embedding(c_in//compression_rate, d_model)
        #self.pos_encoder = PositionalEncoding(d_model, dropout, max_len)
        self.transformer = TST(compression_channels, c_out, seq_len//compression_rate, n_layers=n_layers, d_model=d_model, n_heads=n_heads, d_ff=d_ff, dropout=dropout, act=act)
        self.d_model = d_model

    def forward(self, x):
        x = self.conv(x)
        x = self.dropout1(x)
        #x = random_convolution(x, c_in=1, c_out=1, max_kernel_size=1000, mix = False, p = 0.8)
        #x = self.encoder(x) * math.sqrt(self.d_model)
        #x = self.pos_encoder(x)
        x = self.transformer(x)
#        x = x.mean(dim=1)
#        x = self.classifier(x)
        return x

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()